# Project SQL: Data Discovery with the Chinook Database

Welcome to your first major hands-on data exploration project. In this assignment, you will be stepping into the role of a data analyst querying the Chinook database: a sample dataset representing a digital media store, complete with information about artists, albums, tracks, customers, and business sales.

The core objective of this project is to transition you from simply writing code to thinking like a data scientist. To do that, our journey follows a deliberate, four-fold progression:



## Phase 0: Examine the Dataset (Meta)

SQL in sqlite3

In [182]:
import sqlite3 as db
import pandas as pd
import contextlib

In [183]:
%%capture
%%bash
apt-get update
apt-get install -y sqlite3

In [184]:
! sqlite3 --help

Usage: sqlite3 [OPTIONS] [FILENAME [SQL]]
FILENAME is the name of an SQLite database. A new database is created
if the file does not previously exist. Defaults to :memory:.
OPTIONS include:
   --                   treat no subsequent arguments as options
   -A ARGS...           run ".archive ARGS" and exit
   -append              append the database to the end of the file
   -ascii               set output mode to 'ascii'
   -bail                stop after hitting an error
   -batch               force batch I/O
   -box                 set output mode to 'box'
   -column              set output mode to 'column'
   -cmd COMMAND         run "COMMAND" before reading stdin
   -csv                 set output mode to 'csv'
   -deserialize         open the database using sqlite3_deserialize()
   -echo                print inputs before execution
   -init FILENAME       read/process named file
   -[no]header          turn headers on or off
   -help                show this message
   -html    

In [185]:
! sqlite3 --version

3.45.1 2024-01-30 16:01:20 e876e51a0ed5c5b3126f52e532044363a014bc594cfefa87ffb5b82257ccalt1 (64-bit)


In [186]:
# Download the dataset
%%bash
[ -f chinook.zip ] ||
  curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip
unzip -l chinook.zip

Archive:  chinook.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   884736  2015-11-29 10:53   chinook.db
---------                     -------
   884736                     1 file


In [187]:
!ls -l

total 1168
-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 305596 Nov 16 04:40 chinook.zip
drwxr-xr-x 1 root root   4096 Nov 12 16:40 sample_data


In [188]:
!rm -f chinook.db

In [189]:
!ls -l

total 304
-rw-r--r-- 1 root root 305596 Nov 16 04:40 chinook.zip
drwxr-xr-x 1 root root   4096 Nov 12 16:40 sample_data


In [190]:
!unzip -u chinook.zip

Archive:  chinook.zip
  inflating: chinook.db              


In [191]:
!ls -l

total 1168
-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 305596 Nov 16 04:40 chinook.zip
drwxr-xr-x 1 root root   4096 Nov 12 16:40 sample_data


In [192]:
# Get a list of the tables in the database
%%script sqlite3 --column --header chinook.db
.tables

albums          employees       invoices        playlists     
artists         genres          media_types     tracks        
customers       invoice_items   playlist_track


In [193]:
# Show the schema for the entire database
%%script sqlite3 --column --header chinook.db
.schema

CREATE TABLE IF NOT EXISTS "albums"
(
    [AlbumId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Title] NVARCHAR(160)  NOT NULL,
    [ArtistId] INTEGER  NOT NULL,
    FOREIGN KEY ([ArtistId]) REFERENCES "artists" ([ArtistId]) 
		ON DELETE NO ACTION ON UPDATE NO ACTION
);
CREATE TABLE sqlite_sequence(name,seq);
CREATE TABLE IF NOT EXISTS "artists"
(
    [ArtistId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);
CREATE TABLE IF NOT EXISTS "customers"
(
    [CustomerId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [FirstName] NVARCHAR(40)  NOT NULL,
    [LastName] NVARCHAR(20)  NOT NULL,
    [Company] NVARCHAR(80),
    [Address] NVARCHAR(70),
    [City] NVARCHAR(40),
    [State] NVARCHAR(40),
    [Country] NVARCHAR(40),
    [PostalCode] NVARCHAR(10),
    [Phone] NVARCHAR(24),
    [Fax] NVARCHAR(24),
    [Email] NVARCHAR(60)  NOT NULL,
    [SupportRepId] INTEGER,
    FOREIGN KEY ([SupportRepId]) REFERENCES "employees" ([EmployeeId]) 
		ON DELETE NO ACTION ON 

## Phase 1. Master the Mechanics (Get Familiar with SQL)



Before you can extract deep insights, you need to speak the language. You will start by practicing the fundamental mechanics of SQL using the SQL statements below: learning how to filter data, join tables, group metrics, and aggregate numbers. Think of this as learning how to move around the data environment smoothly.  As a starting point, pick a single table and answer the "How many ...?" questions using SQL.  Then repeat the process for each table.



### 1. Artist

In [194]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from artists
limit 5

ArtistId  Name             
--------  -----------------
1         AC/DC            
2         Accept           
3         Aerosmith        
4         Alanis Morissette
5         Alice In Chains  


In [195]:
# Select specific columns
%%script sqlite3 --column --header chinook.db

SELECT
    ArtistId,
    Name
FROM artists
LIMIT 10;

ArtistId  Name                
--------  --------------------
1         AC/DC               
2         Accept              
3         Aerosmith           
4         Alanis Morissette   
5         Alice In Chains     
6         Antônio Carlos Jobim
7         Apocalyptica        
8         Audioslave          
9         BackBeat            
10        Billy Cobham        


In [196]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
-- select * from pragma_table_info('artists') -- must have 'artists', not artists
PRAGMA table_info(artists);

cid  name      type           notnull  dflt_value  pk
---  --------  -------------  -------  ----------  --
0    ArtistId  INTEGER        1                    1 
1    Name      NVARCHAR(120)  0                    0 


In [197]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('artists')
WHERE pk > 0;

name      type     pk
--------  -------  --
ArtistId  INTEGER  1 


In [198]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('artists');

In [199]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('artists')

total_columns
-------------
2            


In [200]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_artists
from artists

total_artists
-------------
275          


In [201]:
# Count rows where name is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_names
FROM artists
WHERE Name IS NULL;

null_names
----------
0         


In [202]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ArtistId IS NULL THEN 1 ELSE 0 END) AS null_artist_ids,
    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS null_names
FROM artists;

total_rows  null_artist_ids  null_names
----------  ---------------  ----------
275         0                0         


In [203]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ArtistId IS NULL OR Name IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM artists;

total_rows  rows_with_nulls
----------  ---------------
275         0              


In [204]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ArtistId IS NULL THEN 1 ELSE 0 END) AS null_artist_ids,
    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS null_names,
    SUM(
        CASE
            WHEN ArtistId IS NULL OR Name IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM artists;

total_rows  null_artist_ids  null_names  rows_with_nulls
----------  ---------------  ----------  ---------------
275         0                0           0              


#### Filtering and Searching Records: WHERE, AND, OR, LIKE, BETWEEN, LIMIT

In [205]:
# WHERE and LIKE (%)
%%script sqlite3 --column --header chinook.db
SELECT ArtistId, Name
FROM artists
WHERE Name LIKE 'A%';

ArtistId  Name                                                                                 
--------  -------------------------------------------------------------------------------------
1         AC/DC                                                                                
2         Accept                                                                               
3         Aerosmith                                                                            
4         Alanis Morissette                                                                    
5         Alice In Chains                                                                      
6         Antônio Carlos Jobim                                                                 
7         Apocalyptica                                                                         
8         Audioslave                                                                           
26        Azymuth                       

In [206]:
# WHERE and LIKE (_%)
%%script sqlite3 --column --header chinook.db
SELECT ArtistId, Name
FROM artists
WHERE Name LIKE 'A_%';

ArtistId  Name                                                                                 
--------  -------------------------------------------------------------------------------------
1         AC/DC                                                                                
2         Accept                                                                               
3         Aerosmith                                                                            
4         Alanis Morissette                                                                    
5         Alice In Chains                                                                      
6         Antônio Carlos Jobim                                                                 
7         Apocalyptica                                                                         
8         Audioslave                                                                           
26        Azymuth                       

In [207]:
# AND and BETWEEN
%%script sqlite3 --column --header chinook.db
SELECT ArtistId, Name
FROM artists
WHERE ArtistId BETWEEN 1 AND 50
  AND Name LIKE '%H%';

ArtistId  Name                           
--------  -------------------------------
3         Aerosmith                      
5         Alice In Chains                
10        Billy Cobham                   
12        Black Sabbath                  
17        Chico Buarque                  
18        Chico Science & Nação Zumbi    
23        Frank Zappa & Captain Beefheart
26        Azymuth                        
44        Kid Abelha                     
47        Hermeto Pascoal                
48        Barão Vermelho                 


In [208]:
# OR
%%script sqlite3 --column --header chinook.db
SELECT ArtistId, Name
FROM artists
WHERE ArtistId = 1
   OR ArtistId = 2
   OR ArtistId = 3;

ArtistId  Name     
--------  ---------
1         AC/DC    
2         Accept   
3         Aerosmith


In [209]:
# LIMIT
%%script sqlite3 --column --header chinook.db
SELECT *
FROM artists
LIMIT 5;

ArtistId  Name             
--------  -----------------
1         AC/DC            
2         Accept           
3         Aerosmith        
4         Alanis Morissette
5         Alice In Chains  


In [210]:
# Count unique artist names
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT Name) AS unique_artist_names
FROM artists;

unique_artist_names
-------------------
275                


In [211]:
# ORDER BY
%%script sqlite3 --column --header chinook.db
SELECT
    ArtistId,
    Name
FROM artists
ORDER BY Name ASC;

ArtistId  Name                                                                                 
--------  -------------------------------------------------------------------------------------
43        A Cor Do Som                                                                         
1         AC/DC                                                                                
230       Aaron Copland & London Symphony Orchestra                                            
202       Aaron Goldberg                                                                       
214       Academy of St. Martin in the Fields & Sir Neville Marriner                           
215       Academy of St. Martin in the Fields Chamber Ensemble & Sir Neville Marriner          
222       Academy of St. Martin in the Fields, John Birch, Sir Neville Marriner & Sylvia McNair
257       Academy of St. Martin in the Fields, Sir Neville Marriner & Thurston Dart            
239       Academy of St. Martin in the F

In [212]:
# GROUP BY
%%script sqlite3 --column --header chinook.db

SELECT
    Name,
    COUNT(*) AS occurrences
FROM artists
GROUP BY Name
ORDER BY occurrences DESC;

Name                                                                                   occurrences
-------------------------------------------------------------------------------------  -----------
Zeca Pagodinho                                                                         1          
Youssou N'Dour                                                                         1          
Yo-Yo Ma                                                                               1          
Yehudi Menuhin                                                                         1          
Xis                                                                                    1          
Wilhelm Kempff                                                                         1          
Whitesnake                                                                             1          
Vinícius E Qurteto Em Cy                                                               1          
Vinícius E

In [213]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT Name, COUNT(*) AS occurrences
FROM artists
GROUP BY Name
ORDER BY Name
LIMIT 10;

Name                                                                                   occurrences
-------------------------------------------------------------------------------------  -----------
A Cor Do Som                                                                           1          
AC/DC                                                                                  1          
Aaron Copland & London Symphony Orchestra                                              1          
Aaron Goldberg                                                                         1          
Academy of St. Martin in the Fields & Sir Neville Marriner                             1          
Academy of St. Martin in the Fields Chamber Ensemble & Sir Neville Marriner            1          
Academy of St. Martin in the Fields, John Birch, Sir Neville Marriner & Sylvia McNair  1          
Academy of St. Martin in the Fields, Sir Neville Marriner & Thurston Dart              1          
Academy of

In [214]:
# Aggregate functions
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS total_artists,
    SUM(ArtistId) AS sum_artist_ids,
    AVG(ArtistId) AS average_artist_id,
    MAX(ArtistId) AS highest_artist_id,
    MIN(ArtistId) AS lowest_artist_id
FROM artists;

total_artists  sum_artist_ids  average_artist_id  highest_artist_id  lowest_artist_id
-------------  --------------  -----------------  -----------------  ----------------
275            37950           138.0              275                1               


In [215]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    ArtistId,
    Name,
    COUNT(*) AS occurrences
FROM artists
GROUP BY ArtistId, Name
HAVING COUNT(*) > 1;

In [216]:
# Count excess duplicate rows
%%script sqlite3 --column --header chinook.db

SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM artists
    GROUP BY ArtistId, Name
    HAVING COUNT(*) > 1
);

duplicate_rows
--------------
0             


In [217]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(ArtistId) FROM artists

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(ArtistId) FROM artists)
)

SELECT id AS missing_artist_id
FROM expected_ids
WHERE id NOT IN (
    SELECT ArtistId FROM artists
);

### 2. Albums

In [218]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from albums
limit 5

AlbumId  Title                                  ArtistId
-------  -------------------------------------  --------
1        For Those About To Rock We Salute You  1       
2        Balls to the Wall                      2       
3        Restless and Wild                      2       
4        Let There Be Rock                      1       
5        Big Ones                               3       


In [219]:
# Select specific columns
%%script sqlite3 --column --header chinook.db
SELECT
    AlbumId,
    Title
FROM albums
LIMIT 5;

AlbumId  Title                                
-------  -------------------------------------
1        For Those About To Rock We Salute You
2        Balls to the Wall                    
3        Restless and Wild                    
4        Let There Be Rock                    
5        Big Ones                             


In [220]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
PRAGMA table_info(albums);

cid  name      type           notnull  dflt_value  pk
---  --------  -------------  -------  ----------  --
0    AlbumId   INTEGER        1                    1 
1    Title     NVARCHAR(160)  1                    0 
2    ArtistId  INTEGER        1                    0 


In [221]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('albums')
WHERE pk > 0;

name     type     pk
-------  -------  --
AlbumId  INTEGER  1 


In [222]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('albums');

id  seq  table    from      to        on_update  on_delete  match
--  ---  -------  --------  --------  ---------  ---------  -----
0   0    artists  ArtistId  ArtistId  NO ACTION  NO ACTION  NONE 


In [223]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('albums')

total_columns
-------------
3            


In [224]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_albums
from albums

total_albums
------------
347         


In [225]:
# Count rows where name is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_titles
FROM albums
WHERE title IS NULL;

null_titles
-----------
0          


In [226]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN albumId IS NULL THEN 1 ELSE 0 END) AS null_album_ids,
    SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END) AS null_titles
FROM albums;

total_rows  null_album_ids  null_titles
----------  --------------  -----------
347         0               0          


In [227]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN albumId IS NULL OR title IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM albums;

total_rows  rows_with_nulls
----------  ---------------
347         0              


In [228]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN albumId IS NULL THEN 1 ELSE 0 END) AS null_album_ids,
    SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END) AS null_titles,
    SUM(
        CASE
            WHEN albumId IS NULL OR title IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM albums;

total_rows  null_album_ids  null_titles  rows_with_nulls
----------  --------------  -----------  ---------------
347         0               0            0              


#### Filtering and Searching Records: WHERE, AND, OR, LIKE, BETWEEN, LIMIT

In [229]:
# WHERE and LIKE (%)
%%script sqlite3 --column --header chinook.db
SELECT albumId, title
FROM albums
WHERE title LIKE 'A%';

AlbumId  Title                                                    
-------  ---------------------------------------------------------
10       Audioslave                                               
14       Alcohol Fueled Brewtality Live! [Disc 1]                 
15       Alcohol Fueled Brewtality Live! [Disc 2]                 
24       Afrociberdelia                                           
26       Acústico MTV [Live]                                      
29       Axé Bahia 2001                                           
74       Album Of The Year                                        
75       Angel Dust                                               
85       As Canções de Eu Tu Eles                                 
89       American Idiot                                           
90       Appetite for Destruction                                 
94       A Matter of Life and Death                               
95       A Real Dead One                                      

In [230]:
# WHERE and LIKE (_%)
%%script sqlite3 --column --header chinook.db
SELECT albumId, title
FROM albums
WHERE title LIKE 'A_%';

AlbumId  Title                                                    
-------  ---------------------------------------------------------
10       Audioslave                                               
14       Alcohol Fueled Brewtality Live! [Disc 1]                 
15       Alcohol Fueled Brewtality Live! [Disc 2]                 
24       Afrociberdelia                                           
26       Acústico MTV [Live]                                      
29       Axé Bahia 2001                                           
74       Album Of The Year                                        
75       Angel Dust                                               
85       As Canções de Eu Tu Eles                                 
89       American Idiot                                           
90       Appetite for Destruction                                 
94       A Matter of Life and Death                               
95       A Real Dead One                                      

In [231]:
# AND and BETWEEN
%%script sqlite3 --column --header chinook.db
SELECT albumId, title
FROM albums
WHERE albumId BETWEEN 1 AND 50
  AND title LIKE '%Z%';

AlbumId  Title                 
-------  ----------------------
22       Sozinho Remix Ao Vivo 
33       Chill: Brazil (Disc 1)
34       Chill: Brazil (Disc 2)


In [232]:
# OR
%%script sqlite3 --column --header chinook.db
SELECT albumId, title
FROM albums
WHERE albumId = 1
   OR albumId = 2
   OR albumId = 3;

AlbumId  Title                                
-------  -------------------------------------
1        For Those About To Rock We Salute You
2        Balls to the Wall                    
3        Restless and Wild                    


In [233]:
# LIMIT
%%script sqlite3 --column --header chinook.db
SELECT *
FROM albums
LIMIT 5;

AlbumId  Title                                  ArtistId
-------  -------------------------------------  --------
1        For Those About To Rock We Salute You  1       
2        Balls to the Wall                      2       
3        Restless and Wild                      2       
4        Let There Be Rock                      1       
5        Big Ones                               3       


In [234]:
# Count unique album titles
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT title) AS unique_album_titles
FROM albums;

unique_album_titles
-------------------
347                


In [235]:
# ORDER BY
%%script sqlite3 --column --header chinook.db
SELECT
    albumId,
    title
FROM albums
ORDER BY title ASC;

AlbumId  Title                                                                                          
-------  -----------------------------------------------------------------------------------------------
156      ...And Justice For All                                                                         
257      20th Century Masters - The Millennium Collection: The Best of Scorpions                        
296      A Copland Celebration, Vol. I                                                                  
94       A Matter of Life and Death                                                                     
95       A Real Dead One                                                                                
96       A Real Live One                                                                                
285      A Soprano Inspired                                                                             
139      A TempestadeTempestade Ou O Livro Dos Dias    

In [236]:
# GROUP BY
%%script sqlite3 --column --header chinook.db
SELECT
    title,
    COUNT(*) AS occurrences
FROM albums
GROUP BY title
ORDER BY occurrences DESC;

Title                                                                                            occurrences
-----------------------------------------------------------------------------------------------  -----------
[1997] Black Light Syndrome                                                                      1          
Zooropa                                                                                          1          
Worlds                                                                                           1          
Weill: The Seven Deadly Sins                                                                     1          
Warner 25 Anos                                                                                   1          
War                                                                                              1          
Walking Into Clarksdale                                                                          1          
Wagner: Favourite O

In [237]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT title, COUNT(*) AS occurrences
FROM albums
GROUP BY title
ORDER BY title
LIMIT 10;

Title                                                                    occurrences
-----------------------------------------------------------------------  -----------
...And Justice For All                                                   1          
20th Century Masters - The Millennium Collection: The Best of Scorpions  1          
A Copland Celebration, Vol. I                                            1          
A Matter of Life and Death                                               1          
A Real Dead One                                                          1          
A Real Live One                                                          1          
A Soprano Inspired                                                       1          
A TempestadeTempestade Ou O Livro Dos Dias                               1          
A-Sides                                                                  1          
Ace Of Spades                                                    

In [238]:
# Aggregate functions
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS total_albums,
    SUM(albumId) AS sum_album_ids,
    AVG(albumId) AS average_album_id,
    MAX(albumId) AS highest_album_id,
    MIN(albumId) AS lowest_album_id
FROM albums;

total_albums  sum_album_ids  average_album_id  highest_album_id  lowest_album_id
------------  -------------  ----------------  ----------------  ---------------
347           60378          174.0             347               1              


In [239]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    albumId,
    title,
    COUNT(*) AS occurrences
FROM albums
GROUP BY albumId, title
HAVING COUNT(*) > 1;

In [240]:
# Count excess duplicate rows
%%script sqlite3 --column --header chinook.db

SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM albums
    GROUP BY albumId, title
    HAVING COUNT(*) > 1
);

duplicate_rows
--------------
0             


In [241]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(albumId) FROM albums

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(albumId) FROM albums)
)

SELECT id AS missing_album_id
FROM expected_ids
WHERE id NOT IN (
    SELECT albumId FROM albums
);

In [242]:
# Foreign key check
%%script sqlite3 --column --header chinook.db
SELECT
    AlbumId,
    title,
    ArtistId
FROM albums
WHERE NOT EXISTS (
    SELECT 1
    FROM artists
    WHERE artists.ArtistId = albums.ArtistId
);

### 3. Tracks

In [243]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from tracks
limit 5

TrackId  Name                                     AlbumId  MediaTypeId  GenreId  Composer                                                                Milliseconds  Bytes     UnitPrice
-------  ---------------------------------------  -------  -----------  -------  ----------------------------------------------------------------------  ------------  --------  ---------
1        For Those About To Rock (We Salute You)  1        1            1        Angus Young, Malcolm Young, Brian Johnson                               343719        11170334  0.99     
2        Balls to the Wall                        2        2            1                                                                                342562        5510424   0.99     
3        Fast As a Shark                          3        2            1        F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman                     230619        3990994   0.99     
4        Restless and Wild                        3        2     

In [244]:
# Select specific columns
%%script sqlite3 --column --header chinook.db
SELECT
    trackId,
    name
FROM tracks
LIMIT 5;

TrackId  Name                                   
-------  ---------------------------------------
1        For Those About To Rock (We Salute You)
2        Balls to the Wall                      
3        Fast As a Shark                        
4        Restless and Wild                      
5        Princess of the Dawn                   


In [245]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
PRAGMA table_info(tracks);

cid  name          type           notnull  dflt_value  pk
---  ------------  -------------  -------  ----------  --
0    TrackId       INTEGER        1                    1 
1    Name          NVARCHAR(200)  1                    0 
2    AlbumId       INTEGER        0                    0 
3    MediaTypeId   INTEGER        1                    0 
4    GenreId       INTEGER        0                    0 
5    Composer      NVARCHAR(220)  0                    0 
6    Milliseconds  INTEGER        1                    0 
7    Bytes         INTEGER        0                    0 
8    UnitPrice     NUMERIC(10,2)  1                    0 


In [246]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('tracks')
WHERE pk > 0;

name     type     pk
-------  -------  --
TrackId  INTEGER  1 


In [247]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('tracks');

id  seq  table        from         to           on_update  on_delete  match
--  ---  -----------  -----------  -----------  ---------  ---------  -----
0   0    media_types  MediaTypeId  MediaTypeId  NO ACTION  NO ACTION  NONE 
1   0    genres       GenreId      GenreId      NO ACTION  NO ACTION  NONE 
2   0    albums       AlbumId      AlbumId      NO ACTION  NO ACTION  NONE 


In [248]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('tracks')

total_columns
-------------
9            


In [249]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_tracks
from tracks

total_tracks
------------
3503        


In [250]:
# Count rows where name is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_names
FROM tracks
WHERE name IS NULL;

null_names
----------
0         


In [251]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN trackId IS NULL THEN 1 ELSE 0 END) AS null_track_ids,
    SUM(CASE WHEN name IS NULL THEN 1 ELSE 0 END) AS null_names
FROM tracks;

total_rows  null_track_ids  null_names
----------  --------------  ----------
3503        0               0         


In [252]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN trackId IS NULL OR name IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM tracks;

total_rows  rows_with_nulls
----------  ---------------
3503        0              


In [253]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN trackId IS NULL THEN 1 ELSE 0 END) AS null_track_ids,
    SUM(CASE WHEN name IS NULL THEN 1 ELSE 0 END) AS null_names,
    SUM(
        CASE
            WHEN trackId IS NULL OR name IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM tracks;

total_rows  null_track_ids  null_names  rows_with_nulls
----------  --------------  ----------  ---------------
3503        0               0           0              


#### Filtering and Searching Records: WHERE, AND, OR, LIKE, BETWEEN, LIMIT

In [254]:
# WHERE and LIKE (%)
%%script sqlite3 --column --header chinook.db
SELECT trackId, name
FROM tracks
WHERE name LIKE 'A%';

TrackId  Name                                                            
-------  ----------------------------------------------------------------
30       Amazing                                                         
36       Angel                                                           
38       All I Really Want                                               
72       Angela                                                          
134      All For You                                                     
139      A.N.D.R.O.T.A.Z.                                                
170      A Statistic                                                     
220      A Luz De Tieta                                                  
221      Atrás Da Verd-E-Rosa Só Não Vai Quem Já Morreu                  
231      Atras Da Porta                                                  
235      Apesar De Você                                                  
236      A Banda                      

In [255]:
# WHERE and LIKE (_%)
%%script sqlite3 --column --header chinook.db
SELECT trackId, name
FROM tracks
WHERE name LIKE 'A_%';

TrackId  Name                                                            
-------  ----------------------------------------------------------------
30       Amazing                                                         
36       Angel                                                           
38       All I Really Want                                               
72       Angela                                                          
134      All For You                                                     
139      A.N.D.R.O.T.A.Z.                                                
170      A Statistic                                                     
220      A Luz De Tieta                                                  
221      Atrás Da Verd-E-Rosa Só Não Vai Quem Já Morreu                  
231      Atras Da Porta                                                  
235      Apesar De Você                                                  
236      A Banda                      

In [256]:
# AND and BETWEEN
%%script sqlite3 --column --header chinook.db
SELECT trackId, name
FROM tracks
WHERE trackId BETWEEN 1 AND 50
  AND name LIKE '%Z%';

TrackId  Name   
-------  -------
30       Amazing
34       Crazy  


In [257]:
# OR
%%script sqlite3 --column --header chinook.db
SELECT trackId, name
FROM tracks
WHERE trackId = 1
   OR trackId = 2
   OR trackId = 3;

TrackId  Name                                   
-------  ---------------------------------------
1        For Those About To Rock (We Salute You)
2        Balls to the Wall                      
3        Fast As a Shark                        


In [258]:
# LIMIT
%%script sqlite3 --column --header chinook.db
SELECT *
FROM tracks
LIMIT 5;

TrackId  Name                                     AlbumId  MediaTypeId  GenreId  Composer                                                                Milliseconds  Bytes     UnitPrice
-------  ---------------------------------------  -------  -----------  -------  ----------------------------------------------------------------------  ------------  --------  ---------
1        For Those About To Rock (We Salute You)  1        1            1        Angus Young, Malcolm Young, Brian Johnson                               343719        11170334  0.99     
2        Balls to the Wall                        2        2            1                                                                                342562        5510424   0.99     
3        Fast As a Shark                          3        2            1        F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman                     230619        3990994   0.99     
4        Restless and Wild                        3        2     

In [259]:
# Count unique track names
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT name) AS unique_track_names
FROM tracks;

unique_track_names
------------------
3257              


In [260]:
# ORDER BY
%%script sqlite3 --column --header chinook.db
SELECT
    trackId,
    name
FROM tracks
ORDER BY name ASC;

TrackId  Name                                                                                                                       
-------  ---------------------------------------------------------------------------------------------------------------------------
3027     "40"                                                                                                                       
2918     "?"                                                                                                                        
3412     "Eine Kleine Nachtmusik" Serenade In G, K. 525: I. Allegro                                                                 
109      #1 Zero                                                                                                                    
3254     #9 Dream                                                                                                                   
602      'Round Midnight                                             

In [262]:
# GROUP BY
%%script sqlite3 --column --header chinook.db
SELECT
    name,
    COUNT(*) AS TrackCount
FROM tracks
GROUP BY name
ORDER BY TrackCount DESC;

Name                                                                                                                         TrackCount
---------------------------------------------------------------------------------------------------------------------------  ----------
Wrathchild                                                                                                                   5         
The Trooper                                                                                                                  5         
The Number Of The Beast                                                                                                      5         
Iron Maiden                                                                                                                  5         
Hallowed Be Thy Name                                                                                                         5         
2 Minutes To Midnight                           

In [263]:
%%script sqlite3 --column --header chinook.db

SELECT
    AlbumId,
    COUNT(*) AS TrackCount
FROM tracks
GROUP BY AlbumId
ORDER BY TrackCount DESC;

AlbumId  TrackCount
-------  ----------
141      57        
23       34        
73       30        
229      26        
230      25        
251      25        
83       24        
231      24        
253      24        
24       23        
228      23        
255      23        
51       22        
224      22        
250      22        
39       21        
167      21        
37       20        
54       20        
55       20        
115      20        
221      20        
227      19        
248      19        
258      19        
21       18        
72       18        
102      18        
145      18        
146      18        
202      18        
211      18        
213      18        
18       17        
26       17        
33       17        
34       17        
36       17        
120      17        
162      17        
163      17        
185      17        
193      17        
203      17        
237      17        
243      17        
259      17        
261      17        


In [264]:
%%script sqlite3 --column --header chinook.db

SELECT
    GenreId,
    COUNT(*) AS TrackCount
FROM tracks
GROUP BY GenreId
ORDER BY TrackCount DESC;

GenreId  TrackCount
-------  ----------
1        1297      
7        579       
3        374       
4        332       
2        130       
19       93        
6        81        
24       74        
21       64        
14       61        
8        58        
9        48        
10       43        
23       40        
17       35        
15       30        
13       28        
16       28        
20       26        
12       24        
22       17        
11       15        
18       13        
5        12        
25       1         


In [265]:
%%script sqlite3 --column --header chinook.db

SELECT
    MediaTypeId,
    COUNT(*) AS TrackCount
FROM tracks
GROUP BY MediaTypeId
ORDER BY TrackCount DESC;

MediaTypeId  TrackCount
-----------  ----------
1            3034      
2            237       
3            214       
5            11        
4            7         


In [266]:
%%script sqlite3 --column --header chinook.db

SELECT
    Composer,
    COUNT(*) AS TrackCount
FROM tracks
GROUP BY Composer
ORDER BY TrackCount DESC
LIMIT 15;

Composer                                        TrackCount
----------------------------------------------  ----------
                                                978       
Steve Harris                                    80        
U2                                              44        
Jagger/Richards                                 35        
Billy Corgan                                    31        
Kurt Cobain                                     26        
Bill Berry-Peter Buck-Mike Mills-Michael Stipe  25        
The Tea Party                                   24        
Miles Davis                                     23        
Gilberto Gil                                    23        
Chris Cornell                                   23        
Chico Science                                   23        
Titãs                                           22        
Renato Russo                                    20        
J.C. Fogerty                                    20      

In [267]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT name, COUNT(*) AS occurrences
FROM tracks
GROUP BY name
ORDER BY name
LIMIT 10;

Name                                                        occurrences
----------------------------------------------------------  -----------
"40"                                                        1          
"?"                                                         1          
"Eine Kleine Nachtmusik" Serenade In G, K. 525: I. Allegro  1          
#1 Zero                                                     1          
#9 Dream                                                    1          
'Round Midnight                                             1          
(Anesthesia) Pulling Teeth                                  1          
(Da Le) Yaleo                                               1          
(I Can't Help) Falling In Love With You                     1          
(Oh) Pretty Woman                                           1          


In [268]:
# Aggregate functions
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS total_tracks,
    SUM(trackId) AS sum_track_ids,
    AVG(trackId) AS average_track_id,
    MAX(trackId) AS highest_track_id,
    MIN(trackId) AS lowest_track_id
FROM tracks;

total_tracks  sum_track_ids  average_track_id  highest_track_id  lowest_track_id
------------  -------------  ----------------  ----------------  ---------------
3503          6137256        1752.0            3503              1              


In [269]:
# Check out float features
%%script sqlite3 --column --header chinook.db

SELECT
    MIN(Milliseconds) AS ShortestTrack,
    MAX(Milliseconds) AS LongestTrack,
    ROUND(AVG(Milliseconds), 2) AS AverageMilliseconds,
    SUM(Milliseconds) AS TotalMilliseconds
FROM tracks;

SELECT
    MIN(Bytes) AS SmallestFile,
    MAX(Bytes) AS LargestFile,
    ROUND(AVG(Bytes), 2) AS AverageBytes
FROM tracks;

SELECT
    MIN(UnitPrice) AS LowestPrice,
    MAX(UnitPrice) AS HighestPrice,
    AVG(UnitPrice) AS AveragePrice
FROM tracks;

ShortestTrack  LongestTrack  AverageMilliseconds  TotalMilliseconds
-------------  ------------  -------------------  -----------------
1071           5286953       393599.21            1378778040       
SmallestFile  LargestFile  AverageBytes
------------  -----------  ------------
38747         1059546140   33510207.07 
LowestPrice  HighestPrice  AveragePrice    
-----------  ------------  ----------------
0.99         1.99          1.05080502426492


In [270]:
# Duration in minutes
%%script sqlite3 --column --header chinook.db

SELECT
    Name,
    ROUND(Milliseconds / 60000.0, 2) AS DurationMinutes
FROM tracks
ORDER BY Milliseconds DESC
LIMIT 10;

Name                         DurationMinutes
---------------------------  ---------------
Occupation / Precipice       88.12          
Through a Looking Glass      84.81          
Greetings from Earth, Pt. 1  49.34          
The Man With Nine Lives      49.28          
Battlestar Galactica, Pt. 2  49.27          
Battlestar Galactica, Pt. 1  49.21          
Murder On the Rising Star    48.93          
Battlestar Galactica, Pt. 3  48.8           
Take the Celestra            48.79          
Fire In Space                48.78          


In [271]:
# Frequency of track by price
%%script sqlite3 --column --header chinook.db

SELECT
    UnitPrice,
    COUNT(*) AS TrackCount
FROM tracks
GROUP BY UnitPrice
ORDER BY UnitPrice;

UnitPrice  TrackCount
---------  ----------
0.99       3290      
1.99       213       


In [272]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    trackId,
    name,
    COUNT(*) AS occurrences
FROM tracks
GROUP BY trackId, name
HAVING COUNT(*) > 1;

In [273]:
# Count excess duplicate rows
%%script sqlite3 --column --header chinook.db

SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM tracks
    GROUP BY trackId, name
    HAVING COUNT(*) > 1
);

duplicate_rows
--------------
0             


In [274]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(trackId) FROM tracks

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(trackId) FROM tracks)
)

SELECT id AS missing_track_id
FROM expected_ids
WHERE id NOT IN (
    SELECT trackId FROM tracks
);

In [275]:
# Foreign key check: AlbumId
%%script sqlite3 --column --header chinook.db

SELECT
    TrackId,
    Name,
    AlbumId
FROM tracks
WHERE AlbumId IS NOT NULL
AND NOT EXISTS (
    SELECT 1
    FROM albums
    WHERE albums.AlbumId = tracks.AlbumId
);

In [276]:
# Foreign key check: MediaTypeId
%%script sqlite3 --column --header chinook.db

SELECT
    TrackId,
    Name,
    MediaTypeId
FROM tracks
WHERE MediaTypeId IS NOT NULL
AND NOT EXISTS (
    SELECT 1
    FROM media_types
    WHERE media_types.MediaTypeId = tracks.MediaTypeId
);

In [277]:
# Foreign key check: GenreId
%%script sqlite3 --column --header chinook.db

SELECT
    TrackId,
    Name,
    GenreId
FROM tracks
WHERE GenreId IS NOT NULL
AND NOT EXISTS (
    SELECT 1
    FROM genres
    WHERE genres.GenreId = tracks.GenreId
);

In [278]:
# Alternative: use joins
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS InvalidAlbumReferences
FROM tracks t
LEFT JOIN albums a ON t.AlbumId = a.AlbumId
WHERE t.AlbumId IS NOT NULL
AND a.AlbumId IS NULL;

SELECT
    COUNT(*) AS InvalidMediaTypeReferences
FROM tracks t
LEFT JOIN media_types m ON t.MediaTypeId = m.MediaTypeId
WHERE t.MediaTypeId IS NOT NULL
AND m.MediaTypeId IS NULL;

SELECT
    COUNT(*) AS InvalidGenreReferences
FROM tracks t
LEFT JOIN genres g ON t.GenreId = g.GenreId
WHERE t.GenreId IS NOT NULL
AND g.GenreId IS NULL;

InvalidAlbumReferences
----------------------
0                     
InvalidMediaTypeReferences
--------------------------
0                         
InvalidGenreReferences
----------------------
0                     


## Phase 2. Map the Terrain (Get Familiar with the Data)



Data never exists in a vacuum. By writing queries, you will uncover the shape, structure, and quirks of the Chinook dataset. Again, using the SQL statements below, you'll learn how the tables relate to one another, what a single row represents in different contexts, and where specific types of information live.  Here's where you should get familiar with the ERD and join tables in various ways.



## Phase 3. Answer Guided Questions



Once you know the syntax and understand the data structure, you will put your skills to work solving specific business problems. Below are sets of explicit questions to answer using SQL. This phase mimics standard reporting: you are handed a specific prompt, and your job is to translate that prompt into an accurate, data-driven answer.

Here are some practice questions:
- https://github.com/LucasMcL/15-sql_queries_02-chinook/blob/master/README.md
- [Using the R language]( https://rpubs.com/enext777/636199 )
- [Search Google]( https://www.google.com/search?q=chinook+database+questions )
- Ask AI to generate some questions for you.  Be sure to mention the Chinook data set.




## Phase 4. Ask and Answer Your Own Questions


This final phase is the most critical part of the project. The true hallmark of a great data scientist isn't just answering the questions you're handed; it's having the curiosity and intuition to formulate your own questions.

You are tasked with looking at the Chinook dataset, generating your own original hypotheses or business questions, and writing the SQL required to answer them.

Demonstrate that you know SQL fundamentals by posting a question for each SQL statement below and then using the SQL statement to answer you question.

You will use the material from this phase for the presentation.



## Why This Matters: The Path to Your Capstone


Learning to ask your own questions is the exact skill you will rely on for your final capstone project. In your capstone, you will choose a dataset from your own field of interest. No one will hand you a list of prompts or a guide telling you what to look for.

Use this project as your sandbox. Mastering the ability to interrogate a database now will give you the framework you need to discover meaningful, independent insights in your capstone later.



## Chinook data set

See the lecture on SQLite3 using the Chinook data set to set up the software, database, and tables, as well as for the links to ancillary information about the data set.


In [279]:
# Install the sqlite package for Ubuntu
# Download the Chinook sqlite database


## Use SQL queries to pull specific information

Do NOT pull all the data and then filter using DataFrame methods.  The initial purpose is to get familiar with SQL.




### Basic Queries


#### SELECT (with * and with column names)


#### WHERE


#### AND


#### OR


#### LIKE (with % or _ wildcard)


#### BETWEEN


#### LIMIT



### Sorting and Grouping


#### ORDER BY


#### DISTINCT


#### GROUP BY



### Aggregates


#### MAX


#### MIN


#### SUM


#### AVG


#### COUNT



## Make some plots

Make some cool plots to go with your data. Write SQL queries to get ONLY the information you need for each plot. (Don't pull ALL the data and then just plot a few columns.)



## EXTRA CREDIT:
* Use a CTE
* Use a query that joins two or more tables.
* Make a model to see if you can predict something ( e.g. GNB classifier, linear regression )
* Come up with something else cool to do with your data
